# 리포트 56 — TX·RX·표적 배치와 β·앙각·원거리장이 유효창을 연다

> ### 한 일
> **조명원과 패시브 수신기를 지상에 고정하고 표적을 공중에 두는 자유공간 배치를 좌표로 못 박은 뒤, 그 배치에서 결과가 성립하는 창을 세 축으로 재었다.**

### 결과
1. 베이스라인 500 m [^1] · 표적 고도 60 m [^2] · 장면 방위 90° [^3] 에서 푼다.
2. solve 판 R90(기체 s1000plus [^4] · 모드 W1 [^5]) 의 바이스태틱 각은 2.95° [^6] 라 준모노스태틱이고, σ 는 이등분선 방향의 모노스태틱 값을 쓴다.
3. 푸는 게이트는 β ≤ 90° (`src/freespace_scene.py:81` `BETA_VALID_MAX_DEG` 를 `beta_gate()` 가 그대로 쓴다 — `:398`) 이고, 상반성 rms 잔차가 β ≤ 45° 안에서 2.57 dB [^7], β 60~90° 에서 4.02 dB [^8] 다.
4. 장면 방위 72 [^9]방위 전수 스윕에서 σ 를 고정한 순수기하의 R90 span 은 W1 에서 0.48% [^10] 이고 세 팔을 다 세면 0.17 [^11] ~ 0.48% [^12] 다 — 세 팔(W1·L1·G1) 모두 φ=90° 가 최솟값이라 이 배치의 φ 는 보수적인 끝이다.
5. σ 격자의 앙각 행은 9 개 [^13] 이고 최솟값이 -20° [^14] 다 — 조회의 일부가 경계 행으로 클램프된다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 좌표 | 조명원(TX)과 패시브 수신기(RX)를 지상에 고정하고, 표적을 두 점의 중점에서 수평거리 `d` 만큼 떨어진 공중에 둔다 — `src/freespace_scene.py:72`, 기하 함수 `:117` |
| 바이스태틱 게이트와 상반성 창 | 푸는 게이트는 β ≤ 90° (`src/freespace_scene.py:81` `BETA_VALID_MAX_DEG` 를 `beta_gate()` 가 그대로 쓴다 — `:398`) 다. 상반성 rms 잔차는 β ≤ 45° 행에서 최대 2.57 dB [^7] 이고 β 60~90° 행에서 최대 4.02 dB [^8] 라, 두 각을 나란히 실어 그 차이의 크기를 숫자로 둔다 |
| σ 조회 | 이등분선 방향의 모노스태틱 값을 격자에서 조회한다 — `src/experiment_freespace_sigma.py:227` |
| 발표된 격자의 신원 | 이 결과가 실제로 읽은 σ 격자판은 아카이브에 그대로 있고, φ 스윕이 기록한 생성시각과 일치하는 것으로 특정한다(빌드마다 확인한다) |

### 재현

```bash
cd /workspace/sionna
PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_sigma.py
for D in mini5pro mavic4pro matrice4e phantom4 s1000plus; do \
  PYTHONPATH=src ~/.venvs/py312/bin/python src/experiment_freespace_range.py \
    --stage all --mode W1,L1,G1 --drone $D; done
PYTHONPATH=src ~/.venvs/py312/bin/python src/build_part10_results.py
```

| | |
|---|---|
| 출력 | `outputs/report13_freespace.json`, `outputs/verify_freespace.json`, `outputs/sbr_defect_fixes.json`, `outputs/phi_sweep.json` |
| 소요 | σ 격자 · 검지거리 4단계 · 검증 · 스윕을 합쳐 7.8 h [^15] (GPU 2 [^16]장). 이 빌더 자신은 CPU 수 초다 |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| [편 47 «바이스태틱 거리 분해능은 c/B»](47_range-convention.ipynb) | 거리·잡음대역 규약 |

---

## 배치 — TX · RX · 표적을 어디에 두었나

조명원(TX)과 패시브 수신기(RX)를 지상에 고정하고, 표적을 두 점의 중점에서 수평거리 `d` 만큼 떨어진 공중에 둔다. 좌표 상수 `src/freespace_scene.py:72`, 기하 함수 `src/freespace_scene.py:117`.

| 항목 | 값 | 무엇을 정하나 |
|---|---|---|
| 베이스라인 $L$ | 500 m [^1] | β(d) 와 직접파 세기 |
| 표적 고도 | 60 m [^2] | 이등분선 앙각 el |
| 장면 방위 $\varphi$ | 90° [^3] | R1 · R2 의 비 |
| EIRP · 수신이득 · NF | 63 dBm [^17] · 10 dBi [^18] · 5 dB [^19] | 선언 예산 — 잡음바닥과 절대 거리 축 |
| CPI | 0.1 s [^20] | 프레임 수 M = CPI·PRF |
| 기준채널 | full_waveform_capture [^21] | 상관에 쓸 수 있는 에너지 |

## 유효창 — β 와 앙각이 어디까지 열려 있나

solve 판 R90(기체 s1000plus [^4] · 모드 W1 [^5]) 에서 β = 2.95° [^6] 라 준모노스태틱이고, σ 는 이등분선 방향의 모노스태틱 값을 쓴다(`src/experiment_freespace_sigma.py:227`). 그 거리축은 재현 루프의 마지막 기체가 남긴 판이고, 편 60 의 헤드라인 폭은 앵커 비교가능 기체 쪽에서 읽는다([편 60 «앵커 σ 위의 R90 은 비교가능 12칸에서…»](60_r90.ipynb)). 아래 표가 게이트와 창을 가른다.

| 창 | 성립 범위 | 크기 |
|---|---|---|
| 바이스태틱 각 | 푸는 게이트 β ≤ 90° · 상반성 창 β ≤ 45° | 상반성 rms 잔차 β≤45° 2.57 dB [^7] · β 60~90° 4.02 dB [^8] |
| σ 격자 앙각 | el ≥ -20° [^14] (`d` ≥ 126 m [^22]) | 격자 앙각 행 9 개 [^13], solve 판 R90 의 el = -0.26° [^23] (⚠ 이 키는 마지막으로 푼 모드 G1 이 덮어쓴 값이다) |
| β = 45° 지점 (기하만의 함수) | `d` = 602 m [^24] | 그 지점의 SNR(기체 s1000plus · 모드 W1) = 58 dB [^25] |
| 장면 방위 φ | 72 [^9]방위 전수 — 5° 간격의 전 원주 | σ 고정 순수기하 R90 span 0.48% [^10] · 자세평균 0.83% [^26] |

## 앙각 클램프 — 이 창의 열린 끝

같은 스윕이 σ 조회의 앙각도 잰다. 스윕이 읽은 격자(생성 2026-07-29T05:36:31 [^27], 앙각 0~−20°)에서 조회의 4.6% [^28] (φ=90°) ~ 22.5% [^29] (φ=0°) 가 경계 행으로 클램프됐다 — 격자 밖 값을 가장자리 값으로 눌러 붙였다는 뜻이다.

근거리 SNR 천장이 그 조회 위에 서므로, 확장된 앙각 격자 위에서 다시 푸는 일을 다음 단계에 건다.

그 천장 66.81 dB [^30] 은 `d` = 248 m [^31] 에서 서고, 그 자리는 위 표의 β = 45° 지점보다 안쪽이라 게이트 안 · 상반성 창 밖이다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 앙각을 확장한 σ 격자 위에서 R90 과 SNR 천장을 다시 푼다 | φ 축에서 4.6% [^28] ~ 22.5% [^29] 이던 클램프 조회가 격자 안으로 들어오고, 근거리 SNR 천장이 격자 위에 선다 | `src/experiment_freespace_sigma.py` → `--stage solve` |
| β > 45° 의 출사 가시성·대칭화 잔차를 다시 잰다 | 바이스태틱 유효창의 폭이 확정된다 | `benchmark/verify_sbr_defect_fixes.py` → [편 20 «수신 방향 그림자 광선을 켜면 상반성 위반이…»](20_bistatic-exit.ipynb) |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 31개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report13_freespace.json` | `solve.W1.L_m` | 500 |
| [^2] | `outputs/report13_freespace.json` | `solve.W1.alt_m` | 60 |
| [^3] | `outputs/report13_freespace.json` | `solve.W1.phi_deg` | 90 |
| [^4] | `outputs/report13_freespace.json` | `solve.W1.drone` | s1000plus |
| [^5] | `outputs/report13_freespace.json` | `solve.W1.mode` | W1 |
| [^6] | `outputs/report13_freespace.json` | `solve.W1.beta_deg → R90 에서 보간` | (240행 표) (파생) |
| [^7] | `outputs/sbr_defect_fixes.json` | `d2_reciprocity_drone.rows → β≤45 행 최대` | (7행 표) (파생) |
| [^8] | `outputs/sbr_defect_fixes.json` | `d2_reciprocity_drone.rows → β>45 행 최대` | (7행 표) (파생) |
| [^9] | `outputs/phi_sweep.json` | `meta.n_phi` | 72 |
| [^10] | `outputs/phi_sweep.json` | `verdict.claims[2].range_over_phi.constant_sigma_control.W1.span_pct_of_phi90` | 0.4793 |
| [^11] | `outputs/phi_sweep.json` | `verdict.claims[2].range_over_phi.*.span_pct_of_phi90 → 세 팔 최소` | (여러 칸) |
| [^12] | `outputs/phi_sweep.json` | `verdict.claims[2].range_over_phi.*.span_pct_of_phi90 → 세 팔 최대` | (여러 칸) |
| [^13] | `outputs/archive/report13_sigma_grid_pre0803.json` | `meta.el_deg → 길이` | (9행 표) (파생) |
| [^14] | `outputs/archive/report13_sigma_grid_pre0803.json` | `meta.el_deg → 최솟값` | (9행 표) (파생) |
| [^15] | `outputs/report05_derived.json` | `runtime.total_h` | 7.773 |
| [^16] | `outputs/report13_freespace.json` | `meta.gpus` | 2 |
| [^17] | `outputs/report13_freespace.json` | `meta.link_budget.eirp_dbm` | 63 |
| [^18] | `outputs/report13_freespace.json` | `meta.link_budget.rx_gain_dbi` | 10 |
| [^19] | `outputs/report13_freespace.json` | `meta.link_budget.noise_figure_db` | 5 |
| [^20] | `outputs/report13_freespace.json` | `solve.W1.T_cpi_s` | 0.1 |
| [^21] | `outputs/report13_freespace.json` | `meta.link_budget.power_normalization.canonical_reference` | full_waveform_capture |
| [^22] | `outputs/report13_freespace.json` | `solve.W1.el_look_deg → el=−20° 보간` | (240행 표) (파생) |
| [^23] | `outputs/report13_freespace.json` | `meta.ranges_el_look_deg` | -0.2626 |
| [^24] | `outputs/report13_freespace.json` | `solve.W1.beta_deg → β=45° 보간` | (240행 표) (파생) |
| [^25] | `outputs/report13_freespace.json` | `solve.W1.snr_d_db → d=β45 에서 보간` | (240행 표) (파생) |
| [^26] | `outputs/phi_sweep.json` | `verdict.claims[2].range_over_phi.aspect_averaged.W1.span_pct_of_phi90` | 0.8331 |
| [^27] | `outputs/phi_sweep.json` | `meta.sigma_file_generated` | 2026-07-29T05:36:31 |
| [^28] | `outputs/phi_sweep.json` | `geometry.rows[18].frac_el_outside_sigma_grid` | 0.04583 |
| [^29] | `outputs/phi_sweep.json` | `geometry.rows[0].frac_el_outside_sigma_grid` | 0.225 |
| [^30] | `outputs/report13_freespace.json` | `solve.W1.snr_ceiling_db` | 66.81 |
| [^31] | `outputs/report13_freespace.json` | `solve.W1.snr_peak_d_m` | 248.2 |